# Phone → notebook → detection

1. Run `make ui` and open the printed HTTPS URL on the phone. Select **Audio**, **Video**, or both.
2. Select this project's **.venv** as the notebook's Python kernel in VS Code.
3. Run **Connect**, then choose the **gesture** or **voice** example below. You do not need to run both.

The phone captures media; the Mac runs the models locally. No cloud account or API key is needed.
The first model load downloads public model weights into `.local/models`. Inference runs on the gateway.
If setting up another computer, install `gateway/requirements-notebook.txt` into its Python environment first.

## 1. Connect

Keep the gateway running in its terminal. Use the admin password (`admin` by default) to access monitored media.
The notebook runs on the same Mac/Pi as the gateway, so it uses localhost.

In [ ]:
import json, time
from getpass import getpass
from io import BytesIO
from pathlib import Path
from urllib.request import Request, build_opener, HTTPCookieProcessor, urlopen
from http.cookiejar import CookieJar
from PIL import Image
from IPython.display import Audio, clear_output, display

GATEWAY = "http://localhost:8080"
ADMIN_PASSWORD = getpass("Admin password (Enter for admin): ") or "admin"
connection = build_opener(HTTPCookieProcessor(CookieJar()))
login = Request(GATEWAY + "/api/admin/login", data=json.dumps({"password": ADMIN_PASSWORD}).encode(), headers={"Content-Type": "application/json"})
with connection.open(login, timeout=10) as response:
    assert response.status == 200
ADMIN_PASSWORD = None

# Monitoring uses the authenticated admin session cookie.
def get(path):
    request = Request(GATEWAY + path)
    with connection.open(request, timeout=10) as response:
        return response.read(), response.headers

def get_frame():
    data, _ = get("/api/frame.jpg")
    return Image.open(BytesIO(data)).convert("RGB") if data else None

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "gateway/server.py").exists())
MODELS = PROJECT / ".local/models"
MODELS.mkdir(parents=True, exist_ok=True)
print(json.loads(get("/api/status")[0]))

## 2. Image example: recognize a hand gesture

Select **Video** on the phone. Run the detection cell, then hold **Hold to capture** while showing a hand gesture.
The model is Google's [MediaPipe Gesture Recognizer](https://developers.google.com/edge/mediapipe/solutions/vision/gesture_recognizer/python).
First load downloads the official model (about 8 MB). Run the model cell once, then the detection cell whenever needed.

In [ ]:
import mediapipe as mp
import numpy as np

model_path = MODELS / "gesture_recognizer.task"
if not model_path.exists():
    model_url = "https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task"
    with urlopen(model_url, timeout=60) as response:
        model_path.write_bytes(response.read())

options = mp.tasks.vision.GestureRecognizerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(model_path)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
)
print("Gesture model ready.")

In [ ]:
# Run for 10 seconds. Interrupt the cell to stop sooner.
with mp.tasks.vision.GestureRecognizer.create_from_options(options) as recognizer:
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        image = get_frame()
        if image is None:
            print("Waiting for video. Select Video and hold the capture button on the phone.")
            time.sleep(1)
            continue
        rgb = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(image))
        result = recognizer.recognize_for_video(rgb, time.monotonic_ns() // 1_000_000)
        clear_output(wait=True)
        display(image.resize((480, round(image.height * 480 / image.width))))
        if result.gestures:
            gesture = result.gestures[0][0]
            print(f"{gesture.category_name}: {gesture.score:.0%}")
        else:
            print("No hand detected")
        time.sleep(0.1)

## 3. Audio example: listen, then recognize speech

Select **Audio** on the phone. Keep the capture button held while speaking and running the audio cells below.
The gateway exposes the latest five seconds as **16 kHz mono PCM16 WAV**, held only in memory.
This cell lets you listen to what actually reached the gateway; press Play in the output.

In [ ]:
audio, _ = get("/api/audio.wav")
if audio:
    display(Audio(audio))
else:
    print("No recent audio. Select Audio and hold the capture button on the phone.")

### Voice → text

This uses [faster-whisper](https://github.com/SYSTRAN/faster-whisper) with the small **tiny** model on the CPU.
First load downloads model weights (about 75 MB). After that, inference runs offline.
The example transcribes short chunks rather than every audio packet; processing speed depends on the computer.

In [ ]:
from faster_whisper import WhisperModel

voice_model = WhisperModel("tiny", device="cpu", compute_type="int8", download_root=str(MODELS))
print("Voice model ready. Say a short sentence, such as 'turn on the light'.")

In [ ]:
# Read only new audio; the sequence number prevents repeating the same samples.
after = json.loads(get("/api/status")[0])["audio_sequence"]
deadline = time.monotonic() + 10
while time.monotonic() < deadline:
    time.sleep(3)
    audio, headers = get(f"/api/audio.wav?after={after}")
    if not audio:
        print("Waiting for microphone audio…")
        continue
    after = int(headers["X-Audio-Sequence"])
    if headers.get("X-Audio-Gap") == "true":
        print("Skipped older audio to keep up with the live stream.")
    segments, _ = voice_model.transcribe(BytesIO(audio), beam_size=1, vad_filter=True)
    text = " ".join(segment.text.strip() for segment in segments)
    print(text or "No speech detected")

## What to change next

Replace the recognition call with your own model; the web app and phone connection stay the same.
`get_frame()` returns a PIL RGB image. `get("/api/audio.wav?after=...")` returns fresh WAV audio and the next sequence number.

These cells display predictions. The gateway also continues its normal command workflow during capture.
The gesture example polls preview images (up to 960 × 720), not every original camera frame.
Audio chunks can split a sentence; a production voice-command pipeline needs speech boundaries and overlap.
Only the latest five seconds are buffered, so slow consumers skip old audio instead of building a delay.
The app captures for up to 15 seconds per hold. The notebook reads live media while the button is held.